# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates step-by-step exploration and processing of the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record sets and their fields/columns with their @ids

print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet: {record_set.id} (name: {record_set.name})")
    print("  Fields/Columns:")
    for field in record_set.fields:
        print(f"    - {field.id} (name: {field.name}, data_type: {field.data_type})")
    # some implementations may nest columns, print if present
    if hasattr(record_set, 'columns') and record_set.columns:
        print("  Columns:")
        for column in record_set.columns:
            print(f"    - {column.id} (name: {column.name}, data_type: {column.data_type})")
print("\nNote: Use the `@id` string (shown as record_set.id, field.id, column.id) for all extraction tasks.")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the `recordSet` and field `@id`s from the overview above.

In [ ]:
# Extract data from the main record set

# Choose main record set by @id (update from above cell's printed list if changed)
# For this dataset, the main survey data should typically be the first or main record set.
# We'll find the first record set and its id programmatically for robustness.
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

main_record_set_id = record_sets[0] if len(record_sets) else None
if main_record_set_id:
    print(f"Loaded columns in main record set '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets found!')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data for further analysis. All field/column references are by their `@id` string.

In [ ]:
# Example: filter entries with PHQ-9 total score above a threshold, normalize, and group by gender or education level
# Make sure you set the correct field/column @ids printed from section 2.

record_set_id = main_record_set_id
# Replace with the actual @id of the PHQ-9 total score and group field if available from overview
numeric_field_id = None
group_field_id = None
columns = list(dataframes[record_set_id].columns)

# Try to auto-detect likely PHQ-9 total field by case-insensitive string search
for cid in columns:
    if 'phq' in cid.lower() and ('score' in cid.lower() or 'total' in cid.lower()):
        numeric_field_id = cid
        break
# Fallback: Use the first numeric-looking field
if numeric_field_id is None:
    # Try to detect first float/integer-like field via type inference
    for cid in columns:
        if pd.api.types.is_numeric_dtype(dataframes[record_set_id][cid]):
            numeric_field_id = cid
            break

# Try to guess a group field: gender or education level
for cid in columns:
    if 'gender' in cid.lower():
        group_field_id = cid
        break
if group_field_id is None:
    for cid in columns:
        if 'education' in cid.lower():
            group_field_id = cid
            break

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    threshold = 10
    # Coerce to numeric in case values loaded as strings
    df = dataframes[record_set_id].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouped analysis
    if group_field_id in filtered_df.columns:
        print(f"Grouped by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Average {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distribution of the chosen numeric field, and/or compare by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='skyblue', bins=20)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
This notebook demonstrated how to programmatically explore the FAIR² Mental Health Survey Dataset using the `mlcroissant` package.

- We inspected dataset metadata and record sets by their unique `@id`s.
- Survey data was loaded and basic exploratory statistics and visualizations were computed.
- All code references entities by their `@id` as required for robust, schema-guided analysis.

Continue to investigate further associations and patterns using this workflow.